In [1]:
# ── CELL 1: Setup ─────────────────────────────────────────────
!pip install transformers torch torchvision scikit-learn imbalanced-learn pandas numpy matplotlib wilds -q

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, os, json, warnings, math
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from transformers import CLIPModel, CLIPProcessor
warnings.filterwarnings('ignore')

SEEDS        = [42, 0, 1, 7, 99]
DRO_ETA      = 0.1
N_EPOCHS     = 5
BATCH_SIZE   = 32
ADAMW_LR     = 1e-4
ADAMW_WD     = 1e-2
COLLAPSE_THR = 0.01
MAD_I_GATE   = 0.02
FITZ_TAU_BASE = 0.086
FITZ_TAU_W   = 0.131

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cud

In [2]:
# ── CELL 2: Try canonical Waterbirds, fall back to CUB proxy ──
import subprocess

# Attempt 1: canonical Sagawa et al. dataset from Stanford
CANONICAL_URL = 'https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz'
OUT_DIR = '/kaggle/working/waterbirds_canonical'
meta_path = None
img_root  = None

if not os.path.exists(f'{OUT_DIR}/metadata.csv'):
    os.makedirs(OUT_DIR, exist_ok=True)
    print('Attempting canonical Waterbirds download from Stanford...')
    r = subprocess.run(
        ['wget', '-q', '--timeout=60', '-O', f'{OUT_DIR}/wb.tar.gz', CANONICAL_URL],
        capture_output=True)
    if r.returncode == 0 and os.path.getsize(f'{OUT_DIR}/wb.tar.gz') > 1e6:
        subprocess.run(['tar', '-xzf', f'{OUT_DIR}/wb.tar.gz',
                        '-C', OUT_DIR, '--strip-components=1'], capture_output=True)
        print('Canonical dataset extracted.')
    else:
        print(f'Download failed or too small. stderr: {r.stderr.decode()[:200]}')

# Check if canonical metadata exists
for search_root in [OUT_DIR, '/kaggle/input']:
    for root, dirs, files in os.walk(search_root):
        for f in files:
            if f == 'metadata.csv':
                candidate = os.path.join(root, f)
                df_check = pd.read_csv(candidate, nrows=5)
                if 'y' in df_check.columns and 'place' in df_check.columns:
                    # Verify all four cells exist
                    df_full = pd.read_csv(candidate)
                    cells = set(zip(df_full['y'], df_full['place']))
                    if len(cells) == 4:
                        meta_path = candidate
                        img_root  = root
                        print(f'Valid metadata.csv found: {meta_path}')
                        break
        if meta_path:
            break

USING_CANONICAL = meta_path is not None

if USING_CANONICAL:
    df_meta = pd.read_csv(meta_path)
    print(f'CANONICAL Waterbirds: {len(df_meta):,} rows')
    print('Group distribution:')
    print(df_meta.groupby(['y','place']).size().to_string())
else:
    print('Canonical not available. Using CUB-200 proxy.')
    # Build CUB proxy
    cub_root = None
    for root, dirs, files in os.walk('/kaggle/input'):
        subdirs = os.listdir(root) if os.path.isdir(root) else []
        sp = [d for d in subdirs if d[:3].isdigit() and '.' in d]
        if len(sp) >= 10:
            cub_root = root
            break

    if cub_root is None:
        raise FileNotFoundError('No CUB folders found.')

    species_dirs = sorted([d for d in os.listdir(cub_root)
                           if os.path.isdir(os.path.join(cub_root, d)) and d[0].isdigit()])
    rows = []
    for sp_dir in species_dirs:
        sp_num = int(sp_dir.split('.')[0])
        y      = 0 if sp_num <= 100 else 1
        sp_path = os.path.join(cub_root, sp_dir)
        imgs = sorted([f for f in os.listdir(sp_path)
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])
        n = len(imgs)
        split_idx = int(0.75 * n)
        for k, img_f in enumerate(imgs):
            rows.append({'img_filename': os.path.join(sp_path, img_f),
                         'y': y,
                         'place': y if k < split_idx else 1 - y})
    df_meta = pd.DataFrame(rows)
    from sklearn.model_selection import train_test_split as sk_split
    tr_idx, te_idx = sk_split(df_meta.index, test_size=0.2, random_state=42,
                               stratify=df_meta['y'].astype(str)+'_'+df_meta['place'].astype(str))
    df_meta['split_custom'] = 'train'
    df_meta.loc[te_idx, 'split_custom'] = 'test'
    img_root = cub_root
    print(f'CUB-200 proxy: {len(df_meta):,} rows')
    print('NOTE: place is synthetic. NOT canonical Sagawa et al. Waterbirds.')
    print(df_meta.groupby(['y','place']).size().to_string())

print(f'\nUSING_CANONICAL = {USING_CANONICAL}')

Attempting canonical Waterbirds download from Stanford...
Canonical dataset extracted.
Valid metadata.csv found: /kaggle/working/waterbirds_canonical/metadata.csv
CANONICAL Waterbirds: 11,788 rows
Group distribution:
y  place
0  0        6220
   1        2905
1  0         831
   1        1832

USING_CANONICAL = True


In [3]:
# ── CELL 3: Build train/test split ────────────────────────────
majority_mask = ((df_meta['y']==0)&(df_meta['place']==0)) | \
                ((df_meta['y']==1)&(df_meta['place']==1))
minority_mask = ~majority_mask

df_train = df_meta[majority_mask].copy().reset_index(drop=True)
df_test  = df_meta[minority_mask].copy().reset_index(drop=True)

print(f'TRAIN: {len(df_train):,}  TEST: {len(df_test):,}')
print(f'  landbird+land:   {((df_train.y==0)&(df_train.place==0)).sum()}')
print(f'  waterbird+water: {((df_train.y==1)&(df_train.place==1)).sum()}')
print(f'  landbird+water:  {((df_test.y==0)&(df_test.place==1)).sum()}')
print(f'  waterbird+land:  {((df_test.y==1)&(df_test.place==0)).sum()}')

n_lw = int(((df_test.y==0)&(df_test.place==1)).sum())
n_wl = int(((df_test.y==1)&(df_test.place==0)).sum())
MINORITY_CLASS = 0 if n_lw <= n_wl else 1
CLASS_NAMES = {0:'landbird', 1:'waterbird'}
MINORITY_NAME = CLASS_NAMES[MINORITY_CLASS]

n_minority_test = int((df_test.y==MINORITY_CLASS).sum())
n_total_test    = len(df_test)
nc_ng = n_minority_test / n_total_test
print(f'\nMinority class: {MINORITY_NAME} (y={MINORITY_CLASS})')
print(f'nc/Ng = {n_minority_test}/{n_total_test} = {nc_ng:.4f}')

TRAIN: 8,052  TEST: 3,736
  landbird+land:   6220
  waterbird+water: 1832
  landbird+water:  2905
  waterbird+land:  831

Minority class: waterbird (y=1)
nc/Ng = 831/3736 = 0.2224


In [4]:
# ── CELL 4: Load CLIP, extract features ───────────────────────
print('Loading CLIP ViT-L/14...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
clip_model.eval()
print('CLIP loaded.')

def get_img_path(row):
    p = str(row.get('img_filename', ''))
    if os.path.exists(p):
        return p
    c = os.path.join(img_root, p)
    if os.path.exists(c):
        return c
    return None

def extract_features(df, desc=''):
    feats, labels, groups = [], [], []
    failed = 0
    for i in range(0, len(df), 64):
        batch_df = df.iloc[i:i+64]
        imgs, valid_idx = [], []
        for j, (_, row) in enumerate(batch_df.iterrows()):
            path = get_img_path(row)
            if path:
                try:
                    imgs.append(Image.open(path).convert('RGB'))
                    valid_idx.append(j)
                except:
                    failed += 1
            else:
                failed += 1
        if not imgs:
            continue
        inp = clip_proc(images=imgs, return_tensors='pt', padding=True)
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            out = clip_model.visual_projection(
                clip_model.vision_model(**inp).pooler_output)
            out = out / out.norm(dim=-1, keepdim=True)
        feats.append(out.cpu().numpy())
        vr = batch_df.iloc[valid_idx]
        labels.extend(vr['y'].tolist())
        groups.extend((vr['y'].values * 2 + vr['place'].values).tolist())
        if i % 640 == 0:
            print(f'  {desc}: {i+len(imgs)}/{len(df)} ({failed} failed)')
    print(f'  {desc} done: {sum(len(f) for f in feats)} images')
    return np.vstack(feats), np.array(labels), np.array(groups)

print('Extracting TRAIN...')
TR_F, TR_L, TR_G = extract_features(df_train, 'train')
print('Extracting TEST...')
TE_F, TE_L, TE_G = extract_features(df_test, 'test')
print(f'Train: {TR_F.shape}  Test: {TE_F.shape}')

Loading CLIP ViT-L/14...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded.
Extracting TRAIN...
  train: 64/8052 (0 failed)
  train: 704/8052 (0 failed)
  train: 1344/8052 (0 failed)
  train: 1984/8052 (0 failed)
  train: 2624/8052 (0 failed)
  train: 3264/8052 (0 failed)
  train: 3904/8052 (0 failed)
  train: 4544/8052 (0 failed)
  train: 5184/8052 (0 failed)
  train: 5824/8052 (0 failed)
  train: 6464/8052 (0 failed)
  train: 7104/8052 (0 failed)
  train: 7744/8052 (0 failed)
  train done: 8052 images
Extracting TEST...
  test: 64/3736 (0 failed)
  test: 704/3736 (0 failed)
  test: 1344/3736 (0 failed)
  test: 1984/3736 (0 failed)
  test: 2624/3736 (0 failed)
  test: 3264/3736 (0 failed)
  test done: 3736 images
Train: (8052, 768)  Test: (3736, 768)


In [6]:
# ── CELL 5: MAD geometry helpers + sealed envelope ────────────
def mean_pairwise_cosine(X):
    if len(X) < 2: return 0.0
    if len(X) > 500:
        np.random.seed(42)
        X = X[np.random.choice(len(X), 500, replace=False)]
    G = X @ X.T
    return float(G[np.triu_indices(len(X), k=1)].mean())

def compute_mad(mu_cos, n_min, n_maj_demo, n_min_demo, nc_ng_val):
    if nc_ng_val < MAD_I_GATE:
        return None, None, 'MAD-I'
    mad_base = (1 - mu_cos) / math.log(1 + n_min)
    mad_w    = mad_base * math.log(1 + n_maj_demo / n_min_demo) if n_min_demo > 0 else mad_base
    return mad_base, mad_w, 'MAD-G'

# Score every training cell
tr_place = TR_G - TR_L * 2
print('=== ALL-CELLS MAD (sealed before any DRO run) ===')
print(f'{"Cell":<20} {"n":>6} {"nc/Ng":>7} {"mu_cos":>8} {"MAD_base":>10} {"MAD_W":>10} {"pred"}')
print('-'*80)

cell_scores = {}
for y_val in [0, 1]:
    for p_val in [0, 1]:
        mask = (TR_L == y_val) & (tr_place == p_val)
        n = int(mask.sum())
        if n == 0: continue
        feats = TR_F[mask]
        nc    = n / len(TR_L)
        mu    = mean_pairwise_cosine(feats)
        n_maj = len(TR_L) - n
        mb, mw, stg = compute_mad(mu, n, n_maj, n, nc)
        pb = 'ELEVATED' if mb is not None and mb >= FITZ_TAU_BASE else \
             ('CONTRAIND' if mb is None else 'LOW_RISK')
        pw = 'ELEVATED' if mw is not None and mw >= FITZ_TAU_W else \
             ('CONTRAIND' if mw is None else 'LOW_RISK')
        cell_name = f'y={y_val},p={p_val}'
        cell_scores[cell_name] = {
            'n': n, 'nc_ng': round(nc, 4), 'mu': round(mu, 4),
            'mad_base': round(float(mb), 4) if mb is not None else None,
            'mad_w':    round(float(mw), 4) if mw is not None else None,
            'pred_base': pb, 'pred_w': pw
        }
        mb_s = f'{mb:.4f}' if mb is not None else 'N/A'
        mw_s = f'{mw:.4f}' if mw is not None else 'N/A'
        print(f'{cell_name:<20} {n:>6} {nc:>7.3f} {mu:>8.4f} {mb_s:>10} {mw_s:>10} {pb}')

# Single-cell scores for standard pipeline
min_mask    = (TR_L == MINORITY_CLASS)
mu_cos      = mean_pairwise_cosine(TR_F[min_mask])
n_min_train = int(min_mask.sum())
mad_base, mad_w, stage = compute_mad(mu_cos, n_min_train, len(df_train), len(df_test), nc_ng)
pred_base = 'ELEVATED' if (mad_base is not None and mad_base >= FITZ_TAU_BASE) else \
            ('CONTRAINDICATED' if mad_base is None else 'LOW_RISK')
pred_w_s  = 'ELEVATED' if (mad_w is not None and mad_w >= FITZ_TAU_W) else \
            ('CONTRAINDICATED' if mad_w is None else 'LOW_RISK')

mb_str = f'{mad_base:.4f}' if mad_base is not None else 'N/A'
print(f'\nSingle-cell (minority={MINORITY_NAME}): MAD_base={mb_str}  pred={pred_base}')
print(f'Using canonical dataset: {USING_CANONICAL}')

with open('wb_sweep_sealed.json', 'w') as fh:
    json.dump({
        'cell_scores': cell_scores,
        'using_canonical': USING_CANONICAL,
        'minority_class': MINORITY_NAME,
        'nc_ng': round(float(nc_ng), 4),
        'mad_base': round(float(mad_base), 4) if mad_base is not None else None,
        'mad_w':    round(float(mad_w), 4)    if mad_w    is not None else None,
        'pred_base': pred_base,
        'pred_w':    pred_w_s,
    }, fh, indent=2)
print('\n>>> Sealed. <<<')

=== ALL-CELLS MAD (sealed before any DRO run) ===
Cell                      n   nc/Ng   mu_cos   MAD_base      MAD_W pred
--------------------------------------------------------------------------------
y=0,p=0                6220   0.772   0.6740     0.0373     0.0096 LOW_RISK
y=1,p=1                1832   0.228   0.6519     0.0463     0.0686 LOW_RISK

Single-cell (minority=waterbird): MAD_base=0.0463  pred=LOW_RISK
Using canonical dataset: True

>>> Sealed. <<<


In [7]:
# ── CELL 6: Classifier + Group DRO ────────────────────────────
class LinearClassifier(nn.Module):
    def __init__(self): super().__init__(); self.fc = nn.Linear(768, 2)
    def forward(self, x): return self.fc(x)

def make_loaders(tf, tl, tg, ef, el, eg):
    tr = TensorDataset(torch.from_numpy(tf).float(), torch.from_numpy(tl).long(), torch.from_numpy(tg).long())
    te = TensorDataset(torch.from_numpy(ef).float(), torch.from_numpy(el).long(), torch.from_numpy(eg).long())
    return DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True), DataLoader(te, batch_size=128, shuffle=False)

def evaluate(model, loader):
    model.eval(); logits_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb, _ in loader:
            logits_all.append(model(xb.to(device)).cpu()); labels_all.append(yb)
    logits = torch.cat(logits_all); labels = torch.cat(labels_all).numpy()
    probs  = torch.softmax(logits, 1).numpy(); preds = logits.argmax(1).numpy()
    try: auc = roc_auc_score(labels, probs[:,1])
    except: auc = float('nan')
    pc = {int(c): float(accuracy_score(labels[labels==c], preds[labels==c]))
          for c in np.unique(labels)}
    return float(accuracy_score(labels, preds)), auc, pc

def run_erm(tf, tl, ef, el, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    g = np.zeros(len(tl), dtype=int)
    tr_ld, te_ld = make_loaders(tf, tl, g, ef, el, np.zeros(len(el), dtype=int))
    m = LinearClassifier().to(device)
    opt = optim.AdamW(m.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    crit = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        m.train()
        for xb, yb, _ in tr_ld:
            opt.zero_grad(); crit(m(xb.to(device)), yb.to(device)).backward(); opt.step()
    _, auc, pc = evaluate(m, te_ld); return auc, pc

def run_dro(tf, tl, tg, ef, el, eg, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    tr_ld, te_ld = make_loaders(tf, tl, tg, ef, el, eg)
    m = LinearClassifier().to(device)
    opt = optim.AdamW(m.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    n_groups = len(np.unique(tg))
    q = torch.ones(n_groups).to(device) / n_groups
    # Track weight of the SMALLER training cell
    # smaller cell = MINORITY_CLASS in train (fewer congruent examples if canonical)
    minority_grp_idx = MINORITY_CLASS * 2 + (1 - MINORITY_CLASS)  # group with spurious bg
    wt_trace = []
    for epoch in range(N_EPOCHS):
        m.train()
        for xb, yb, gb in tr_ld:
            xb, yb, gb = xb.to(device), yb.to(device), gb.to(device)
            opt.zero_grad()
            losses = nn.CrossEntropyLoss(reduction='none')(m(xb), yb)
            g_loss = torch.zeros(n_groups).to(device)
            for g in range(n_groups):
                mask = gb == g
                if mask.sum() > 0: g_loss[g] = losses[mask].mean()
            q = q * torch.exp(DRO_ETA * g_loss.detach()); q = q / q.sum()
            (q * g_loss).sum().backward(); opt.step()
        min_w = float(q.min().item())
        wt_trace.append(min_w)
    _, auc, pc = evaluate(m, te_ld)
    collapsed = wt_trace[-1] < COLLAPSE_THR
    return auc, pc, wt_trace[-1], collapsed, wt_trace

print('Utilities defined.')

Utilities defined.


In [8]:
# ── CELL 7: 5-seed main experiment ────────────────────────────
ALL_F = np.vstack([TR_F, TE_F])
ALL_L = np.concatenate([TR_L, TE_L])

def rand_auc(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    idx = np.random.permutation(len(ALL_L)); s = int(0.75*len(idx))
    g = np.zeros(s, dtype=int); ge = np.zeros(len(idx)-s, dtype=int)
    tr_ld, te_ld = make_loaders(ALL_F[idx[:s]], ALL_L[idx[:s]], g,
                                 ALL_F[idx[s:]], ALL_L[idx[s:]], ge)
    m = LinearClassifier().to(device)
    opt = optim.AdamW(m.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    crit = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        m.train()
        for xb, yb, _ in tr_ld:
            opt.zero_grad(); crit(m(xb.to(device)), yb.to(device)).backward(); opt.step()
    _, auc, _ = evaluate(m, te_ld); return auc

records = []
for seed in SEEDS:
    print(f'\n{"="*40}\nSEED {seed}\n{"="*40}')
    ra = rand_auc(seed)
    ba, bpc = run_erm(TR_F, TR_L, TE_F, TE_L, seed)
    da, dpc, mw, col, wt = run_dro(TR_F, TR_L, TR_G, TE_F, TE_L, TE_G, seed)
    sa, spc = run_erm(TR_F, TR_L, TE_F, TE_L, seed)  # placeholder; add SMOTE if needed
    print(f'  Rand AUC={ra:.3f}')
    print(f'  Baseline: AUC={ba:.3f} SGG={ra-ba:.3f} {MINORITY_NAME}={bpc.get(MINORITY_CLASS,0):.3f} {CLASS_NAMES[1-MINORITY_CLASS]}={bpc.get(1-MINORITY_CLASS,0):.3f}')
    print(f'  DRO:      AUC={da:.3f} SGG={ra-da:.3f} {MINORITY_NAME}={dpc.get(MINORITY_CLASS,0):.3f} {CLASS_NAMES[1-MINORITY_CLASS]}={dpc.get(1-MINORITY_CLASS,0):.3f} min_w={mw:.4f} collapse={col}')
    for name, auc, pc, wt_col, cf in [
        ('Baseline', ba, bpc, None, None),
        ('Group_DRO', da, dpc, mw, col),
    ]:
        records.append({
            'seed': seed, 'intervention': name, 'rand_auc': round(ra,4),
            'demo_auc': round(auc,4), 'sgg': round(ra-auc,4),
            'minority_acc': round(pc.get(MINORITY_CLASS, float('nan')),4),
            'majority_acc': round(pc.get(1-MINORITY_CLASS, float('nan')),4),
            'min_weight': round(float(wt_col),4) if wt_col is not None else None,
            'weight_collapse': bool(cf) if cf is not None else None,
            'wt_trace': [round(w,4) for w in wt] if name=='Group_DRO' else None,
        })

pd.DataFrame([{k:v for k,v in r.items() if k!='wt_trace'} for r in records])\
  .to_csv('wb_sweep_results.csv', index=False)
print('\nResults saved.')


SEED 42
  Rand AUC=0.960
  Baseline: AUC=0.619 SGG=0.341 waterbird=0.000 landbird=1.000
  DRO:      AUC=0.578 SGG=0.382 waterbird=0.000 landbird=1.000 min_w=0.0000 collapse=True

SEED 0
  Rand AUC=0.951
  Baseline: AUC=0.615 SGG=0.336 waterbird=0.000 landbird=1.000
  DRO:      AUC=0.570 SGG=0.381 waterbird=0.000 landbird=1.000 min_w=0.0000 collapse=True

SEED 1
  Rand AUC=0.951
  Baseline: AUC=0.638 SGG=0.313 waterbird=0.000 landbird=1.000
  DRO:      AUC=0.602 SGG=0.349 waterbird=0.000 landbird=1.000 min_w=0.0000 collapse=True

SEED 7
  Rand AUC=0.945
  Baseline: AUC=0.625 SGG=0.320 waterbird=0.000 landbird=1.000
  DRO:      AUC=0.585 SGG=0.360 waterbird=0.000 landbird=1.000 min_w=0.0000 collapse=True

SEED 99
  Rand AUC=0.953
  Baseline: AUC=0.616 SGG=0.337 waterbird=0.000 landbird=1.000
  DRO:      AUC=0.573 SGG=0.380 waterbird=0.000 landbird=1.000 min_w=0.0000 collapse=True

Results saved.


In [9]:
# ── CELL 8: EXTENDED sweep — subsample TRAINING minority cell ──
# KEY FIX vs previous notebook: subsample TRAIN minority, not test.
# This changes n_min_train and therefore MAD_base, giving a real
# calibration curve instead of a flat line.
print('\n=== EXTENDED SWEEP — subsampling TRAINING minority cell ===')

# Identify the smaller training cell (the one DRO is over-weighting)
tr_place = TR_G - TR_L * 2
# In CUB proxy both cells ~equal; in canonical Waterbirds waterbird-on-water < landbird-on-land
# Sweep the smaller of the two congruent training cells
cell0_mask = (TR_L == 0) & (tr_place == 0)  # landbird+land
cell1_mask = (TR_L == 1) & (tr_place == 1)  # waterbird+water
smaller_cell = 0 if cell0_mask.sum() <= cell1_mask.sum() else 1
smaller_place = smaller_cell
print(f'Sweeping cell: y={smaller_cell}, place={smaller_place} (n={int((TR_L==smaller_cell).sum() & (tr_place==smaller_place).sum())})')

SWEEP_SEEDS = [42, 0, 1]
# Target training minority counts — go from large down to collapse floor
SWEEP_N = [2000, 1000, 500, 200, 100, 50, 25, 10]

smaller_mask = (TR_L == smaller_cell) & (tr_place == smaller_place)
other_mask   = ~smaller_mask
other_f = TR_F[other_mask]; other_l = TR_L[other_mask]; other_g = TR_G[other_mask]

sweep_records = []
for target_n in SWEEP_N:
    cell_idx = np.where(smaller_mask)[0]
    if target_n >= len(cell_idx):
        sel = cell_idx
    else:
        np.random.seed(42)
        sel = np.random.choice(cell_idx, target_n, replace=False)

    sub_f = np.vstack([TR_F[sel], other_f])
    sub_l = np.concatenate([TR_L[sel], other_l])
    sub_g = np.concatenate([TR_G[sel], other_g])

    actual_n  = len(sel)
    nc_ng_sub = actual_n / len(sub_l)
    mu_sub    = mean_pairwise_cosine(TR_F[sel])
    n_maj_sub = len(sub_l) - actual_n
    mb_sub, mw_sub, stg_sub = compute_mad(mu_sub, actual_n, n_maj_sub, actual_n, nc_ng_sub)
    pred_sub  = 'ELEVATED' if mb_sub is not None and mb_sub >= FITZ_TAU_BASE else \
                ('CONTRAIND' if mb_sub is None else 'LOW_RISK')

    collapses = 0; min_accs = []; maj_accs = []
    for seed in SWEEP_SEEDS:
        da, dpc, mw, col, _ = run_dro(sub_f, sub_l, sub_g, TE_F, TE_L, TE_G, seed)
        collapses += int(col)
        min_accs.append(dpc.get(smaller_cell, float('nan')))
        maj_accs.append(dpc.get(1-smaller_cell, float('nan')))

    col_rate = collapses / len(SWEEP_SEEDS)
    mb_str = f'{mb_sub:.4f}' if mb_sub is not None else 'N/A'
    print(f'  n_train={actual_n:>5}  nc/Ng={nc_ng_sub:.3f}  mu={mu_sub:.4f}  '
          f'MAD_base={mb_str}  pred={pred_sub:<12}  '
          f'collapse={collapses}/{len(SWEEP_SEEDS)}  '
          f'min_acc={np.nanmean(min_accs):.3f}  maj_acc={np.nanmean(maj_accs):.3f}')

    sweep_records.append({
        'n_minority_train': actual_n,
        'nc_ng': round(float(nc_ng_sub), 4),
        'mu_cosine': round(float(mu_sub), 4),
        'mad_base': round(float(mb_sub), 4) if mb_sub is not None else None,
        'mad_w':    round(float(mw_sub), 4) if mw_sub is not None else None,
        'pred_base': pred_sub,
        'collapse_rate': round(col_rate, 3),
        'mean_minority_acc': round(float(np.nanmean(min_accs)), 4),
        'mean_majority_acc': round(float(np.nanmean(maj_accs)), 4),
    })

df_sweep = pd.DataFrame(sweep_records)
df_sweep.to_csv('wb_sweep_extended.csv', index=False)
print('\nExtended sweep saved.')
print(df_sweep.to_string())
print('\nKey question: at what n_train does collapse stop, and does MAD_base cross 0.086 there?')


=== EXTENDED SWEEP — subsampling TRAINING minority cell ===
Sweeping cell: y=1, place=1 (n=1832)
  n_train= 1832  nc/Ng=0.228  mu=0.6519  MAD_base=0.0463  pred=LOW_RISK      collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train= 1000  nc/Ng=0.139  mu=0.6497  MAD_base=0.0507  pred=LOW_RISK      collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=  500  nc/Ng=0.074  mu=0.6519  MAD_base=0.0560  pred=LOW_RISK      collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=  200  nc/Ng=0.031  mu=0.6446  MAD_base=0.0670  pred=LOW_RISK      collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=  100  nc/Ng=0.016  mu=0.6507  MAD_base=N/A  pred=CONTRAIND     collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=   50  nc/Ng=0.008  mu=0.6493  MAD_base=N/A  pred=CONTRAIND     collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=   25  nc/Ng=0.004  mu=0.6305  MAD_base=N/A  pred=CONTRAIND     collapse=3/3  min_acc=0.000  maj_acc=1.000
  n_train=   10  nc/Ng=0.002  mu=0.6335  MAD_base=N/A  pred=CONTRAIND  

In [10]:
# ── CELL 9: Summary ───────────────────────────────────────────
dro_rows   = [r for r in records if r['intervention']=='Group_DRO']
n_collapse = sum(r['weight_collapse'] for r in dro_rows)
mean_min   = np.nanmean([r['minority_acc'] for r in dro_rows])
mean_maj   = np.nanmean([r['majority_acc'] for r in dro_rows])

# Locate collapse boundary from sweep
col_df   = df_sweep[df_sweep['collapse_rate'] >= 0.9]
safe_df  = df_sweep[df_sweep['collapse_rate'] == 0.0]
tau_local = None
if len(col_df) > 0 and len(safe_df) > 0:
    col_mad  = col_df['mad_base'].dropna()
    safe_mad = safe_df['mad_base'].dropna()
    if len(col_mad) > 0 and len(safe_mad) > 0:
        tau_local = round(float((col_mad.min() + safe_mad.max()) / 2), 4)

summary = {
    'using_canonical': USING_CANONICAL,
    'dataset': 'Waterbirds_canonical' if USING_CANONICAL else 'Waterbirds_CUB200_proxy',
    'minority_class': MINORITY_NAME,
    'nc_ng': round(float(nc_ng), 4),
    'mad_base_sealed': round(float(mad_base), 4) if mad_base else None,
    'pred_base_sealed': pred_base,
    'dro_collapses': f'{n_collapse}/{len(dro_rows)}',
    'mean_minority_acc_dro': round(float(mean_min), 4),
    'mean_majority_acc_dro': round(float(mean_maj), 4),
    'local_tau_from_train_sweep': tau_local,
    'fitz_threshold_holds': pred_base == 'ELEVATED' and n_collapse == len(dro_rows),
    'note': ('Canonical Sagawa et al. dataset.' if USING_CANONICAL
             else 'CUB-200 proxy with synthetic place labels — NOT canonical Waterbirds.'),
}

with open('wb_sweep_summary.json','w') as fh:
    json.dump(summary, fh, indent=2)

print('=== FINAL SUMMARY ===')
print(json.dumps(summary, indent=2))
print(f'\nMain experiment: DRO collapse {n_collapse}/{len(dro_rows)}')
print(f'  mean {MINORITY_NAME} acc: {mean_min:.3f}')
print(f'  mean {CLASS_NAMES[1-MINORITY_CLASS]} acc: {mean_maj:.3f}')
print(f'Local tau from training sweep: {tau_local}')
print('\nFiles: wb_sweep_results.csv, wb_sweep_extended.csv, wb_sweep_summary.json, wb_sweep_sealed.json')

=== FINAL SUMMARY ===
{
  "using_canonical": true,
  "dataset": "Waterbirds_canonical",
  "minority_class": "waterbird",
  "nc_ng": 0.2224,
  "mad_base_sealed": 0.0463,
  "pred_base_sealed": "LOW_RISK",
  "dro_collapses": "5/5",
  "mean_minority_acc_dro": 0.0,
  "mean_majority_acc_dro": 1.0,
  "local_tau_from_train_sweep": null,
  "fitz_threshold_holds": false,
  "note": "Canonical Sagawa et al. dataset."
}

Main experiment: DRO collapse 5/5
  mean waterbird acc: 0.000
  mean landbird acc: 1.000
Local tau from training sweep: None

Files: wb_sweep_results.csv, wb_sweep_extended.csv, wb_sweep_summary.json, wb_sweep_sealed.json
